# FarmPriceNepal – AI Model Training
This notebook trains and evaluates the forecasting models for Nepal's fresh produce markets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
import sys

# Add parent dir to path
sys.path.append('..')
from app.services.feature_engineering import create_features

## 1. Load Data
We use the generated synthetic data for training.

In [ ]:
prices_df = pd.read_csv('../data/prices.csv')
weather_df = pd.read_csv('../data/weather.csv')

# Merge data
df = pd.merge(prices_df, weather_df, left_on=['market_id', 'date'], right_on=['location_id', 'date'])
df.drop(columns=['location_id'], inplace=True)
print(f"Loaded {len(df)} records")
df.head()

## 2. Feature Engineering

In [ ]:
featured_df = create_features(df)
print(f"Features created. Shape: {featured_df.shape}")
featured_df.columns

## 3. Train-Test Split (Time-based)

In [ ]:
# Split based on date to avoid data leakage in time-series
featured_df = featured_df.sort_values('date')
split_idx = int(len(featured_df) * 0.8)

train_df = featured_df.iloc[:split_idx]
test_df = featured_df.iloc[split_idx:]

features = [
    'day_of_week', 'month', 'is_weekend', 'is_monsoon', 'is_festival',
    'price_lag_1', 'price_lag_7', 'price_lag_30',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_30',
    'temp_c', 'humidity_pct', 'rainfall_mm', 'rain_x_monsoon'
]
target = 'price_npr'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

## 4. Train XGBoost Model

In [ ]:
model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], early_stopping_rounds=50, verbose=False)

print("Model training complete")

## 5. Evaluate

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"MAE: {mae:.2f} NPR")
print(f"RMSE: {rmse:.2f} NPR")
print(f"MAPE: {mape:.2f}%")

## 6. Feature Importance

In [ ]:
importance = pd.DataFrame({'feature': features, 'importance': model.feature_importances_})
importance = importance.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importance)
plt.title('Feature Importance for Market Price Prediction')
plt.show()

## 7. Save Model

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/primary_price_model.pkl')
print("Model saved to models/primary_price_model.pkl")